## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [12]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_openai import OpenAIEmbeddings

c:\udemy\IA\llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [14]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 43 documents


In [43]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1102.61it/s]


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [15]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [16]:


# Load the existing vector store
vector_store = Chroma(
    persist_directory=DB_NAME, 
    embedding_function=embeddings
)

retriever = vector_store.as_retriever()
retriever.invoke("Who is Rolando?")


[Document(id='b9181234-fda7-4b21-983a-9cb582353a34', metadata={'source': 'Marcela_Doc\\cuento.md', 'doc_type': ''}, page_content='El científico y su asistente.\n\nMi nombre es Rolando, un niño con muchas inquietudes y mucha curiosidad; junto con mi asistente-mascota, El Canicas, trabajamos a marchas forzadas en un proyecto que tiene un sentido muy especial para nosotros.\nPrimero debo aclarar que El Canicas tal vez no sea un asistente normal; para la mayoría de las personas es solo un perro de compañía, para mi es parte de mi equipo y un integrante muy importante.'),
 Document(id='aa61d65d-418d-4ac3-830d-60a220aa9e0a', metadata={'doc_type': '', 'source': 'Marcela_Doc\\cuento.md'}, page_content='El Canicas sin saberlo cuenta con las características necesarias para ser un buen colega en el campo de la ciencia, estos rasgos tan importantes son la paciencia y la curiosidad, virtudes que le dan a un científico o aprendiz de este las bases para que su conocimiento se desarrolle y llegue a al

In [18]:
retriever.invoke("Who is Rolando?")

[Document(id='b9181234-fda7-4b21-983a-9cb582353a34', metadata={'doc_type': '', 'source': 'Marcela_Doc\\cuento.md'}, page_content='El científico y su asistente.\n\nMi nombre es Rolando, un niño con muchas inquietudes y mucha curiosidad; junto con mi asistente-mascota, El Canicas, trabajamos a marchas forzadas en un proyecto que tiene un sentido muy especial para nosotros.\nPrimero debo aclarar que El Canicas tal vez no sea un asistente normal; para la mayoría de las personas es solo un perro de compañía, para mi es parte de mi equipo y un integrante muy importante.'),
 Document(id='aa61d65d-418d-4ac3-830d-60a220aa9e0a', metadata={'source': 'Marcela_Doc\\cuento.md', 'doc_type': ''}, page_content='El Canicas sin saberlo cuenta con las características necesarias para ser un buen colega en el campo de la ciencia, estos rasgos tan importantes son la paciencia y la curiosidad, virtudes que le dan a un científico o aprendiz de este las bases para que su conocimiento se desarrolle y llegue a al

In [48]:
llm.invoke("Who is Rolando?")

AIMessage(content='"Rolando" can refer to various individuals or entities depending on the context. Could you please provide more details or specify which Rolando you are referring to? For example, are you asking about a musician, athlete, fictional character, or someone else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 12, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_b95af6e96a', 'id': 'chatcmpl-ETvVmSNWLma30F7GoWmuMbXibVgZI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f413-3178-7c93-95af-592ac0fd3ada-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens'

## Time to put this together!

In [19]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [20]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [21]:
answer_question("Who is Rolando?", [])

'Rolando is a curious and inquisitive young boy who is passionate about science and exploration. He works on a special project with his loyal pet and assistant, El Canicas, and enjoys balancing his scientific pursuits with fun and play.'

In [22]:
answer_question("Who is Rolando?", [])

'Rolando is a curious and inquisitive young boy who is passionate about science and exploration. He works on a special project with his loyal pet and assistant, El Canicas, and enjoys balancing his scientific pursuits with fun and play.'

## What could possibly come next? 😂

In [23]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!